# Diamond price regression — exploratory notebook

**Historical research notebook:** refactored file paths and cleared previous execution outputs. Run from the project subfolder (the folder containing `data/`). For a clean, reproducible holdout benchmark, run `evaluate.py` instead. Results in this notebook are experimental and are not the reported benchmark.


In [ ]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

# Область работы 1 (библиотеки)

In [ ]:
# Данный блок в области 1 НЕ выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
# запрещается скрывать предупреждения системы
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
# pip install

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import svm, metrics
import scipy.stats as stats
from scipy.stats import iqr
import sklearn


from sklearn.neighbors import  KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, ElasticNet, Ridge

from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
from sklearn.model_selection import validation_curve, train_test_split

from sklearn.metrics import make_scorer, mean_absolute_percentage_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.compose import make_column_transformer,ColumnTransformer,TransformedTargetRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer

from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures,KBinsDiscretizer
from sklearn.preprocessing import MinMaxScaler,Normalizer, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder, QuantileTransformer, PowerTransformer

from sklearn.preprocessing import KBinsDiscretizer
import category_encoders as ce


In [ ]:
#pip install --upgrade category_encoders

In [ ]:
import category_encoders as ce
from category_encoders import TargetEncoder


In [ ]:
def histmaker_for_numeric_df(df):
    """рисует хистплоты для каждого столбца датафрейма. 
    Если в дф-е есть столбцы, которые не нужно рисовать, их стоит удалить """
    #оставим только численные столбцы 
    cond = [np.issubdtype(df[f'{column}'], np.number) for column in df.columns]
    df = df[df.columns[cond]]
    
    sns.set(style='darkgrid', palette='deep')

    array_of_columns_of_df = df.values.T.tolist()

    num_plots = len(df.columns)         #количество графиков
    num_rows = (num_plots - 1) // 3 + 1 #количество строк 
    num_cols = min(num_plots, 3)        #количество столбцов

    fig, axes = plt.subplots(num_rows, num_cols, figsize = (15, 4*num_rows))
    plt.suptitle('Histograms of every column in your df', fontsize = 15)
    for i, ax in enumerate(axes.flat):
        col = (np.random.random (), np.random.random (), np.random.random ()) #случайный увет
        if i < num_plots:
            ax.hist(array_of_columns_of_df[i],
                    bins = 50,
                    edgecolor = 'black', color = col, alpha = 0.7)
            ax.set_title(df.columns[i], weight='bold')
        if i % 3 == 0:
            ax.set_ylabel('count')
    
    #Удаление пустых подграфиков
    for i in range(num_plots, num_rows * num_cols):
        fig.delaxes(axes.flatten()[i])

In [ ]:
sns.set_theme(style="whitegrid", palette="pastel")

# Область работы 2 (поиск  модели .... )

In [ ]:
# Данный блок(и) НЕ выполняются преподавателем в области 2
# блок(и) предназначены для поиска лучшей модели 
# должен быть понятен и очевиден отбор параметров модели
# оставляйте свои комментарии и разъяснения
# 
# Запрещается размещать данные блоки за пределами обасти 2
# Все блоки данной области должны быть выполнены
#
# ЗАПРЕЩАЕТСЯ ИСПОЛЬЗОВАТЬ ТЕСТОВЫЙ НАБОР
#
# Путь к тренировочному набору
# 

path_train = 'data/train.csv' 
df_raw = pd.read_csv(path_train)
print(df_raw.shape)
df_raw

In [ ]:
ind_nan = df_raw[df_raw.eq(0).any(axis=1)].index
df_raw[df_raw.eq(0).any(axis=1)]

In [ ]:
histmaker_for_numeric_df(df_raw)

### *Предобработка данных*

In [ ]:
# разделим категориальные и численные столбцы на разные переменные
cat_features = ['color','clarity','cut','symmetry','polish']
axis_features = ['meas_length','meas_width','meas_depth']
nonaxis_num_features = ['size','depth_percent','table_percent']

num_features = ['size','depth_percent','table_percent','meas_length','meas_width','meas_depth']


------------------------------------------------Есть выбросы - уберем их (сравним два метода)------------------------------------------------

In [ ]:
df_num = df_raw[num_features].copy()
df_num.shape

In [ ]:
#попробуем убрать по правилу 3 сигм
z = np.abs(stats.zscore(df_num))
cleaned_df = df_num[(z<3).all(axis=1)].copy()
df_num.where((z<3).all(axis=1))
cleaned_df.shape

In [ ]:
# совместим обработанный дф с нечисленными столцами
mask = [True if index in cleaned_df.index else False for index in df_raw.index]
df_3std = df_raw[mask].copy()
print(df_3std.shape)

In [ ]:
#попробуем iqr
df_num = df_raw[num_features].copy()
df_num.shape

In [ ]:
df_num = df_raw[num_features].copy()

#найдем квантили
Q1 = df_num.quantile(q=.25)
Q3 = df_num.quantile(q=.75)
IQR = df_num.apply(stats.iqr)


In [ ]:

#оставим только попадающие в iqr
df_iqr = df_num[~((df_num < (Q1-1.5*IQR)) | (df_num > (Q3+1.5*IQR))).any(axis=1)]

#find how many rows are left in the dataframe 
df_iqr.shape


In [ ]:
mask = [True if index in df_iqr.index else False for index in df_raw.index]
df_iqred = df_raw[mask].copy()
print(df_iqred.shape)

In [ ]:
figure, ax = plt.subplots(1, 2, figsize = (15,5))
ax[0].set_title('with 3std');
ax[1].set_title('with iqr');
sns.boxplot(data=df_3std, x = 'color', y = 'total_sales_price', ax = ax[0]);
sns.boxplot(data=df_iqred, x = 'color', y = 'total_sales_price', ax = ax[1]);

как будто бы iqr работает лучше, оставим его. К тому же, распределение не симметрично относительно среднего и из-за этого отбор по 3м сигмам может быть плох

------------------------------------------------Уберем дубликаты------------------------------------------------

In [ ]:
df_iqred.drop_duplicates(inplace=True)
df = df_iqred.copy()
print(df.shape)


------------------------------------------------Нулевые значения------------------------------------------------

In [ ]:
# посмотрим на строки с пустыми нулевыми данными - аэээ они убрались во время обработки выборосов.
# все равно нужно будет обработать 0 значения, так как в тесте мы не можем убрать выбросы
df[df.eq(0).any(axis=1)]

------------------------------------------------Корреляция------------------------------------------------

In [ ]:
dataplot = sns.heatmap(df[num_features].corr(), annot = True)
# ого, size очень коррелирует с параметрами размеров (ого!)
# так как size посчитан по размерам и коррелирует, стоит дропнуть size

In [ ]:
print(df.shape)
df.head()

------------------------------------------------Проверим правильность depth_percent------------------------------------------------

в интернетах была найдена формула *depth percent* 
$$\frac{depth}{0.5(length + width)}* 100 $$

In [ ]:
df_check = df.copy()

df_check['dp2'] = round(100 * df_check.meas_depth / (0.5*(df_check.meas_length + df_check.meas_width)),1)
df_check['pr_diff_d'] = (abs(df_check.depth_percent - df_check.dp2)/ df_check.dp2)*100

In [ ]:
df_check[df_check['pr_diff_d'] > 15]

в одном камне наврали с depth_percent, он отличается от настоящего depth_percent аж на 23%

------------------------------------------------Проверим правильность table_percent------------------------------------------------

в интернетах была найдена формула *table percent* 
$$\frac{width}{0.5(length + depth)}* 100 $$

In [ ]:
# Расчет average_diameter
df_check['average_diameter'] = (df_check['meas_length'] + df_check['meas_width']) / 2

In [ ]:
# Расчет table_percent
df_check['tp2'] = (df_check['meas_depth'] / df_check['average_diameter']) * 100

In [ ]:
df_check['pr_diff_p'] = (abs(df_check.table_percent - df_check.tp2)/ df_check.tp2)*100

In [ ]:
df_check[df_check['pr_diff_p'] > 15]

In [ ]:
dataplot = sns.heatmap(df_check[['dp2','tp2','total_sales_price','size']].corr(), annot = True)

короче говоря, есть камни, у которых наврали в этих процентах, но выкидывать мы их не будем, так как они мало коррелируют с ценой. Скаттерплот так же говорит о том, что между ними нет даже нелинейной связи

------------------------------------------------Проверим правильность size------------------------------------------------

формула из интернетов: 
$$ 
size = Diameter^2 * depth * 0.0061
$$

In [ ]:
df_check = df.copy()
df_check[df_check.meas_length > df_check.meas_width]
#вывело 0 строк, то есть у нас всегда meas_length будет меньше или равен meas_width. Берем meas_width за диаметр

In [ ]:
sz = (df_check['meas_width']**2)*df_check['meas_depth']*0.0061
df_check[((np.abs(sz / df_check['size'])) - 1) > 0.15]

In [ ]:
df[df.eq(0).any(axis=1)]

In [ ]:
df

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['total_sales_price']), df.total_sales_price,
                                                    test_size=0.3)

### *First hyperparameter search*

In [ ]:
def binarization(X):
    return np.where(X == 'Excellent', 1, 0)
def weird_gem(df):
    sz = (df['meas_width']**2)*df['meas_depth']*0.0061
    cond = ((abs(sz / df['size'])) - 1) > 0.15
    df.loc[cond, 'meas_length'] = 0
    #df.loc[cond, 'meas_width'] = 0
    #df.loc[cond, 'meas_depth'] = 0
    #print('weird gems nullified')
    return df

In [ ]:
cat_features = ['cut', 'symmetry', 'polish', 'color', 'clarity']
axis_features = ['meas_length', 'meas_width', 'meas_depth', 'size']
num_features = ['depth_percent', 'table_percent']

In [ ]:


axis_transformer = Pipeline(steps=[
    ('strange', FunctionTransformer(weird_gem, validate=False)),
    ('missing_num', IterativeImputer(missing_values=0, max_iter=10)),
    ('polynom', PolynomialFeatures(2,include_bias=False)),
    ('scaler', StandardScaler())])

num = Pipeline(steps=[
    ('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    ('scaler', StandardScaler())])

clarity_map = [{
    'col' : 'clarity',
    'mapping':{'FL':10,'IF':9, 'VVS1':8, 'VVS2':7,
               'VS1':6, 'VS2':5, 'SI1':4, 'SI2':3, 
               'I1':2, 'I2':1, 'I3':0}
}]

clarity_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy = 'most_frequent')),
    ('ce',ce.OrdinalEncoder(mapping=clarity_map)),
    ('scaler', MinMaxScaler())
    ])    
    
CT = ColumnTransformer([
        ("axis_transformer", axis_transformer, axis_features),
        ("num", num, num_features),
        ("cat", OneHotEncoder(sparse_output=False).set_output(transform="pandas"), cat_features),
        ("ordinal_map", clarity_transformer, ['clarity'])
        ]).set_output(transform='pandas')

display(CT)

ct = CT.fit_transform(X_train)
pd.DataFrame(ct).head()



In [ ]:
axis_transformer = Pipeline(steps = [
    ('strange', FunctionTransformer(weird_gem, validate=False)),
    #('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    ('missing_num', KNNImputer(missing_values=0, n_neighbors=3, weights="uniform")),
    ('binning', KBinsDiscretizer(encode='ordinal')),
    ('poly', PolynomialFeatures(2, include_bias=False)),
])

clarity_map = [{
    'col' : 'clarity',
    'mapping':{'IF':9, 'VVS1':8, 'VVS2':7,
               'VS1':6, 'VS2':5, 'SI1':4, 'SI2':3, 
               'I1':2, 'I2':1, 'I3':0}
}]

clarity_transformer = Pipeline(steps=[
    #('missing_num', KNNImputer(strategy = 'most_frequent')),
    ('ce',ce.OrdinalEncoder(mapping=clarity_map)),
    ('scaler', MinMaxScaler())
    ]) 

num_transformer = Pipeline(steps = [
    ('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    ('binning', KBinsDiscretizer(encode='ordinal')),
])

cat_transformer = Pipeline(steps = [
   ('encoder', OneHotEncoder(sparse_output=False)),
])


CT_bin = ColumnTransformer([
    ('axis', axis_transformer, axis_features),
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
    ('clarity',clarity_transformer, ['clarity'])
]).set_output(transform='pandas')

CT_bin

In [ ]:


model_target =  TransformedTargetRegressor(
                regressor = Pipeline(steps=[
                                ('preproc', CT),
                                ('estimator', Ridge())
                            ]),
                transformer=QuantileTransformer(n_quantiles=1000, output_distribution="normal")
)

model_target.fit(X_train, y_train)



In [ ]:

y_pred = model_target.predict(X_train)
y_pred_test = model_target.predict(X_test)
print(f'train MAPE: {mean_absolute_percentage_error(y_train, y_pred)}')
print(f'test MAPE: {mean_absolute_percentage_error(y_test, y_pred_test)}')

### *Эээээээээ два (вроде лучше)*

In [ ]:
cat_features = ['cut', 'symmetry', 'polish', 'color', 'clarity']
axis_features = ['meas_length', 'meas_width', 'meas_depth', 'size']
num_features = ['depth_percent', 'table_percent']

In [ ]:
def binarization(X):
    return np.where(X == 'Excellent', 1, 0)
def weird_gem(df):
    sz = (df['meas_width']**2)*df['meas_depth']*0.0061
    cond = ((abs(sz / df['size'])) - 1) > 0.15
    df.loc[cond, 'meas_length'] = 0
    #df.loc[cond, 'meas_width'] = 0
    #df.loc[cond, 'meas_depth'] = 0
    #print('weird gems nullified')
    return df

In [ ]:

axis_transformer = Pipeline(steps=[
    ('strange', FunctionTransformer(weird_gem, validate=False)),
    #('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    ('missing_num', KNNImputer(missing_values=0, n_neighbors=3, weights="uniform")),
    #('bin', KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='uniform')), с ним хуже..
    ('polynom', PolynomialFeatures(2, include_bias=False)),
    ('scaler', StandardScaler())])

num = Pipeline(steps=[
    ('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    ('scaler', StandardScaler())])

clarity_map = [{
    'col' : 'clarity',
    'mapping':{'IF':9, 'VVS1':8, 'VVS2':7,
               'VS1':6, 'VS2':5, 'SI1':4, 'SI2':3, 
               'I1':2, 'I2':1, 'I3':0}
}]

clarity_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy = 'most_frequent')),
    ('ce',ce.OrdinalEncoder(mapping=clarity_map)),
    ('scaler', MinMaxScaler())
    ])    
    
CT = ColumnTransformer([
        ("axis_transformer", axis_transformer, axis_features),
        ("num", num, num_features),
        ("cat", OneHotEncoder(sparse_output=False).set_output(transform="pandas"), cat_features),
        ("ordinal_map", clarity_transformer, ['clarity'])
        ]).set_output(transform='pandas')

display(CT)

ct = CT.fit_transform(X_train)


In [ ]:
# ээээ тут было 0.0837 на тесте
model = Pipeline([
                ('model', TransformedTargetRegressor(
                        regressor = Pipeline(steps=[
                                ('preproc', CT),
                                ('estimator', Ridge())
                            ]),
                        transformer=PowerTransformer(),
                        check_inverse = False)
                )])

param_grid = [
    {'model__regressor__estimator':[KNeighborsRegressor()],
     'model__regressor__estimator__n_neighbors': [4,11],
     'model__regressor__preproc__num__scaler': [StandardScaler(), RobustScaler(), MinMaxScaler(), Normalizer()],
     'model__transformer':[PowerTransformer(method = 'box-cox'),QuantileTransformer(n_quantiles=1000, output_distribution="normal")],
     'model__regressor__preproc__cat': [OneHotEncoder(sparse_output=False), TargetEncoder()]
    }
]

scores='neg_mean_absolute_percentage_error'
grid = GridSearchCV(model, n_jobs=1, param_grid=param_grid, scoring=scores)
grid.fit(X_train, y_train)

In [ ]:
grid_result = pd.DataFrame(grid.cv_results_).sort_values(["rank_test_score",'std_test_score']).T
grid_result

In [ ]:
mask = grid_result.index[grid_result.index.str.endswith("_test_score")][:-3]
tabl = grid_result.loc[mask].iloc[:,:10]
sns.boxplot(data=tabl);

grid.best_params_

In [ ]:
final = grid.best_estimator_
final.fit(X_train, y_train)

In [ ]:
y_pred = final.predict(X_train)
y_pred_test = final.predict(X_test)
print(f'train MAPE: {mean_absolute_percentage_error(y_train, y_pred)}')
print(f'test MAPE: {mean_absolute_percentage_error(y_test, y_pred_test)}')

In [ ]:
score = []
#устойчивость модели
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['total_sales_price']), df.total_sales_price, test_size=0.3)
    final.fit(X_train, y_train)
    y_pred_test = final.predict(X_test)
    score.append(mean_absolute_percentage_error(y_test, y_pred_test))
    


In [ ]:
print('mean MAPE', np.array(score).mean())
print('std MAPE', np.array(score).std())
# пойдет

# пока что результат выше - лучший

### *Гридсерч по регрессиям* (уже третий или какой там..)

In [ ]:

axis_transformer = Pipeline(steps=[
    ('strange', FunctionTransformer(weird_gem, validate=False)),
    ('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    #('bin', KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='uniform')), с ним хуже..
    ('polynom', PolynomialFeatures(2,include_bias=False)),
    ('scaler', StandardScaler())])

num = Pipeline(steps=[
    ('missing_num', IterativeImputer(missing_values=0, max_iter=20)),
    ('scaler', StandardScaler())])

clarity_map = [{
    'col' : 'clarity',
    'mapping':{'IF':9, 'VVS1':8, 'VVS2':7,
               'VS1':6, 'VS2':5, 'SI1':4, 'SI2':3, 
               'I1':2, 'I2':1, 'I3':0}
}]

clarity_transformer = Pipeline(steps=[
    ('missing_num', SimpleImputer(strategy = 'most_frequent')),
    ('ce',ce.OrdinalEncoder(mapping=clarity_map)),
    ('scaler', MinMaxScaler())
    ])    
    
CT = ColumnTransformer([
        ("axis_transformer", axis_transformer, axis_features),
        ("num", num, num_features),
        ("cat", OneHotEncoder(sparse_output=False).set_output(transform="pandas"), cat_features),
        ("ordinal_map", clarity_transformer, ['clarity'])
        ]).set_output(transform='pandas')

display(CT)

ct = CT.fit_transform(X_train)


In [ ]:
model = Pipeline([
                ('model', TransformedTargetRegressor(
                        regressor = Pipeline(steps=[
                                ('preproc', CT),
                                ('estimator', Ridge())
                            ]),
                        transformer=PowerTransformer(),
                        check_inverse = False)
                )])

param_grid = [
    {'model__regressor__estimator':[Ridge()],
     'model__regressor__preproc__num__scaler': [StandardScaler(), RobustScaler(), MinMaxScaler(), Normalizer()],
     'model__regressor__estimator__alpha': [0.2, 0.4, 0.6, 0.8, 1],
     'model__transformer':[PowerTransformer(),QuantileTransformer(n_quantiles=1000, output_distribution="uniform"), QuantileTransformer(n_quantiles=1000, output_distribution="normal")],
     'model__regressor__preproc__ordinal_map__ce':[ce.OrdinalEncoder(mapping=clarity_map), TargetEncoder()]
    },
    
    {'model__regressor__estimator':[Lasso()],
     'model__regressor__preproc__num__scaler': [StandardScaler(), RobustScaler(), MinMaxScaler(), Normalizer()],
     'model__regressor__estimator__alpha': [0.2, 0.4, 0.6, 0.8, 1],
     'model__transformer':[PowerTransformer(),QuantileTransformer(n_quantiles=1000, output_distribution="uniform"), QuantileTransformer(n_quantiles=1000, output_distribution="normal")],
     'model__regressor__preproc__ordinal_map__ce':[ce.OrdinalEncoder(mapping=clarity_map), TargetEncoder()]}
]


scores='neg_mean_absolute_percentage_error'
grid = GridSearchCV(model, n_jobs=1, param_grid=param_grid, scoring=scores)
grid.fit(X_train, y_train)

In [ ]:
grid_result = pd.DataFrame(grid.cv_results_).sort_values(["rank_test_score",'std_test_score']).T
grid_result

In [ ]:
mask = grid_result.index[grid_result.index.str.endswith("_test_score")][:-3]
tabl = grid_result.loc[mask].iloc[:,:10]
sns.boxplot(data=tabl,  linewidth=1.5);

grid.best_params_

In [ ]:
final = grid.best_estimator_
final.fit(X_train, y_train)

In [ ]:
y_pred = final.predict(X_train)
y_pred_test = final.predict(X_test)
print(f'train MAPE: {mean_absolute_percentage_error(y_train, y_pred)}')
print(f'test MAPE: {mean_absolute_percentage_error(y_test, y_pred_test)}')

 чет регрессия вообще отстой))))))))0